# Test 06
- model: pplx-embed-v1-0.6b  →  1024-dim

## embeddings structure:

```
laceholders_str = (
        f"Template variables: {', '.join(element['placeholders'])}."
        if element['has_placeholders'] and element['placeholders']
        else "No template variables."
    )

    corpus = f"""Title: {element['title']}

Category: {element['category']} > {element['subcategory']}
Difficulty: {element['difficulty']}
Target model: {element.get('target_model', 'general')}
Tags: {', '.join(element['tags'])}
{placeholders_str}

Content:
{element['content']}
"""
    return corpus
```

In [3]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 1 — Setup
# ═══════════════════════════════════════════════════════════════════════════
import json
import numpy as np
import pandas as pd
from pathlib import Path
from semantic_engine_demo.json_load import load_json
from semantic_engine_demo.search_metrics import compute_ndcg
import arrowspace

ROOT     = Path.cwd().parent
DATA_DIR = ROOT / "data"

# ── Load corpus ────────────────────────────────────────────────────────────
corpus    = load_json(DATA_DIR / "dataset.json")           # list[dict], index == pos
benchmark = load_json(DATA_DIR / "benchmark" / "benchmark_queries_01.json")

# ── Load embeddings produced by pplx-embed-v1-0.6b ────────────────────────
# queries_emb_all.npy  shape: (n_entries, 3, dim)
# profiles order:  0=q_medium  1=q_sentence  2=q_verbose
EMB_PATH        = DATA_DIR / "benchmark" / "queries_emb_all.npy"
queries_emb_all = np.load(EMB_PATH)                       # (n, 3, dim)

PROFILES = ["q_medium", "q_sentence", "q_verbose"]
K        = 10
THRESH   = 0.75

assert len(benchmark) == queries_emb_all.shape[0], \
    f"Mismatch: {len(benchmark)} entries vs {queries_emb_all.shape[0]} embeddings"

print(f"Corpus      : {len(corpus)} docs")
print(f"Benchmark   : {len(benchmark)} entries")
print(f"Embeddings  : {queries_emb_all.shape}  (entries × profiles × dim)")
print(f"Model       : pplx-embed-v1-0.6b  →  {queries_emb_all.shape[-1]}-dim")

Corpus      : 20000 docs
Benchmark   : 1379 entries
Embeddings  : (1379, 3, 1024)  (entries × profiles × dim)
Model       : pplx-embed-v1-0.6b  →  1024-dim


In [4]:
embs_c = np.load(DATA_DIR / "embeddings_06.npy")

In [9]:
from arrowspace import ArrowSpaceBuilder
import time

graph_params = {
    "eps": 1.2,
    "k": 25,
    "topk": 20,
    "p": 2.0,
    "sigma": None,
}

print(f"Graph parameters: {graph_params}")

print("Building ArrowSpace...")
start = time.perf_counter()
aspace, gl = (
    ArrowSpaceBuilder()
    .with_seed(42)
    .with_dims_reduction(enabled=False, eps=None)
    .with_sampling("simple", 1.0)
).build_and_store(graph_params, embs_c)
print(f"Build time: {time.perf_counter() - start:.2f}s")

INFO:arrowspace.builder:Initializing new ArrowSpaceBuilder
DEBUG:arrowspace.builder:Creating ArrowSpaceBuilder with default parameters
INFO:arrowspace.builder:Setting custom clustering seed: 42
INFO:arrowspace.builder:Configuring inline sampling: Simple(1)
INFO:arrowspace.builder:Configuring lambda graph: eps=1.2, k=25, p=2, sigma=None
DEBUG:arrowspace.builder:Lambda graph will use raw item magnitudes for normalization
INFO:arrowspace.builder:Setting sparsity check flag: false
INFO:arrowspace.builder:Enabling persistence at: /home/tommaso/code_base/semantic_engine_demo/notebooks/storage
INFO:arrowspace.builder:Building ArrowSpace from 20000 items with 1024 features
DEBUG:arrowspace.builder:Build configuration: eps=1.2, k=25, p=2, sigma=None, normalise=false, synthesis=Median


Graph parameters: {'eps': 1.2, 'k': 25, 'topk': 20, 'p': 2.0, 'sigma': None}
Building ArrowSpace...


DEBUG:arrowspace.builder:raw-input saved
DEBUG:arrowspace.builder:Standard clustering path (F=1024 ≤ 2048)
INFO:arrowspace.builder:EigenMaps::start_clustering: N=20000 items, F=1024 features
DEBUG:arrowspace.builder:Creating ArrowSpace with taumode: Median
DEBUG:arrowspace.builder:Using Simple sampler with ratio 1.00
INFO:arrowspace.sampling:Simple random sampler with keep rate 100.0%
INFO:arrowspace.builder:Computing clustering parameters (heuristic or manual override)
INFO:arrowspace.clustering:Computing optimal K for clustering: N=20000, F=1024
DEBUG:arrowspace.clustering:Two-NN mean ratio: 1.0917, estimated ID: 11
DEBUG:arrowspace.clustering:Intrinsic dimension estimate: 11
DEBUG:arrowspace.clustering:Testing K in range [45, 55] with step 2
DEBUG:arrowspace.clustering:K=51: CH=1.0111, penalized=-280.8253
DEBUG:arrowspace.clustering:K=53: CH=1.0827, penalized=-291.8061
DEBUG:arrowspace.clustering:K=47: CH=1.0082, penalized=-258.7234
DEBUG:arrowspace.clustering:K=55: CH=0.9997, penal

Build time: 206.54s


In [11]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 2 — Search + Evaluation  (fixed)
# ═══════════════════════════════════════════════════════════════════════════

SUBSET_N        = 200      # ← None = full benchmark
SUBSET_SEED     = 54
K               = 10
THRESH          = 0.75
MIN_CAT_ENTRIES = 3        # minimum entries per category to show in Report 2

rng_sub  = np.random.default_rng(SUBSET_SEED)
all_idx  = np.arange(len(benchmark))
idx_pool = (rng_sub.choice(all_idx, size=SUBSET_N, replace=False)
            if SUBSET_N and SUBSET_N < len(benchmark)
            else all_idx)

print(f"Running on {len(idx_pool)}/{len(benchmark)} entries  "
      f"(seed={SUBSET_SEED})  ×  {len(PROFILES)} profiles  "
      f"= {len(idx_pool) * len(PROFILES)} searches")

# ── Helpers (both fixed) ───────────────────────────────────────────────────

def _rank_of(results, tier1_set, k):
    """1-based rank of first tier1 hit within top-k only. None if not found."""
    for i, (doc_id, _) in enumerate(results[:k], 1):   # ← capped at k
        if doc_id in tier1_set:
            return i
    return None

def _recall_tier2(results, tier2_set, k):
    """Fraction of retrievable tier2 docs found in top-k.
    Normalises by min(k, |tier2|) — the maximum achievable — not category_size."""
    if not tier2_set:
        return np.nan
    max_retrievable = min(k, len(tier2_set))            # ← correct denominator
    top_k = {doc_id for doc_id, _ in results[:k]}
    return len(top_k & tier2_set) / max_retrievable

# ── Main loop ──────────────────────────────────────────────────────────────
rows = []

for idx in idx_pool:
    entry     = benchmark[idx]
    tier1     = set(entry["relevant_pos"]["tier1"])
    tier2_raw = set(entry["relevant_pos"]["tier2"]) - tier1
    ref       = [(d, 2.0) for d in tier1] + [(d, 1.0) for d in tier2_raw]

    for p_idx, profile in enumerate(PROFILES):
        emb     = queries_emb_all[idx, p_idx]
        results = aspace.search(emb, gl, THRESH)
        scores  = [s for _, s in results]

        rank = _rank_of(results, tier1, K)              # ← K passed explicitly

        rows.append({
            "entry_id"       : entry["entry_id"],
            "source_title"   : entry["source_title"][:45],
            "source_category": entry["source_category"],
            "category_size"  : entry["category_size"],
            "profile"        : profile,
            "query"          : str(entry["queries"].get(profile, ""))[:65],
            "hit"            : rank is not None,        # now truly Hit@K
            "rank"           : rank,
            "mrr"            : 1.0 / rank if rank else 0.0,
            "ndcg"           : float(compute_ndcg(results, ref, k=K)),
            "recall_tier2"   : _recall_tier2(results, tier2_raw, K),
            "top1_score"     : scores[0] if scores else np.nan,
            "score_gap"      : (scores[0] - scores[K-1]) if len(scores) >= K else np.nan,
        })

df      = pd.DataFrame(rows)
n_total = len(df)
SEP     = "━" * 65

# ── REPORT 1: per-profile ──────────────────────────────────────────────────
print(SEP)
print("  PER-PROFILE")
print(SEP)
display(
    df.groupby("profile")
      .agg(hit_rate   = ("hit",          "mean"),
           mean_mrr   = ("mrr",          "mean"),
           mean_ndcg  = ("ndcg",         "mean"),
           mean_rec2  = ("recall_tier2", "mean"),
           mean_gap   = ("score_gap",    "mean"),
           n          = ("entry_id",     "count"))
      .reindex(PROFILES).round(3)
)

# ── REPORT 2: per-category (min entries filter applied) ───────────────────
print(SEP)
print(f"  PER-CATEGORY  (≥{MIN_CAT_ENTRIES} entries, worst → best NDCG)")
print(SEP)
cat_df = (
    df.groupby("source_category")
      .agg(hit_rate  = ("hit",      "mean"),
           mean_mrr  = ("mrr",      "mean"),
           mean_ndcg = ("ndcg",     "mean"),
           n_entries = ("entry_id", lambda x: x.nunique()))
      .round(3)
      .query("n_entries >= @MIN_CAT_ENTRIES")     # ← noise filter
      .sort_values("mean_ndcg")
)
print(f"  Showing {len(cat_df)} categories (filtered from "
      f"{df['source_category'].nunique()} total)")
display(cat_df)

# ── REPORT 3: headline ─────────────────────────────────────────────────────
print(SEP)
print(f"  HEADLINE  —  pplx-embed-v1-0.6b + ArrowSpace  "
      f"[subset {len(idx_pool)}, seed={SUBSET_SEED}]")
print(SEP)
r2 = df["recall_tier2"].dropna()
print(f"  Hit@{K}          : {df['hit'].mean():.3f}  ({int(df['hit'].sum())}/{n_total})")
print(f"  Mean MRR       : {df['mrr'].mean():.3f}")
print(f"  Mean NDCG@{K}   : {df['ndcg'].mean():.3f}")
print(f"  Recall@tier2   : {r2.mean():.3f}  ({len(r2)} entries with tier2 docs)")
print(f"  Mean score gap : {df['score_gap'].mean():.4f}")

# ── REPORT 4: worst 15 entries ────────────────────────────────────────────
print(SEP)
print("  WORST 15 ENTRIES  (avg NDCG across profiles)")
print(SEP)
display(
    df.groupby(["entry_id", "source_title", "source_category"])
      .agg(mean_ndcg = ("ndcg",          "mean"),
           mean_mrr  = ("mrr",           "mean"),
           hit_rate  = ("hit",           "mean"),
           cat_size  = ("category_size", "first"))
      .reset_index()
      .sort_values("mean_ndcg")
      .head(15)
      .reset_index(drop=True)
      .round(3)
)

# ── REPORT 5: profile robustness ──────────────────────────────────────────
print(SEP)
print("  PROFILE ROBUSTNESS  (NDCG delta vs q_medium baseline)")
print(SEP)
pivot  = (df.pivot_table(index="entry_id", columns="profile",
                         values="ndcg", aggfunc="mean")
            .reindex(columns=PROFILES))
deltas = pivot.sub(pivot["q_medium"], axis=0)
print(deltas.mean().round(4).to_string())
print("\n  + = better than q_medium   − = worse than q_medium")

# ── EXPORT ─────────────────────────────────────────────────────────────────
label = f"n{len(idx_pool)}_s{SUBSET_SEED}"
OUT   = DATA_DIR / "benchmark" / f"eval_pplx06_{label}.csv"
df.to_csv(OUT, index=False)
print(f"\nSaved → {OUT}")

INFO:arrowspace.core:Lambda-aware search: k=20
DEBUG:arrowspace.core:Query vector dimension: 1024, lambda: 0.373777


Running on 200/1379 entries  (seed=54)  ×  3 profiles  = 600 searches


DEBUG:arrowspace.core:Search completed, returning 20 results
INFO:arrowspace.core:Lambda-aware search: k=20
DEBUG:arrowspace.core:Query vector dimension: 1024, lambda: 0.000009
DEBUG:arrowspace.core:Search completed, returning 20 results
INFO:arrowspace.core:Lambda-aware search: k=20
DEBUG:arrowspace.core:Query vector dimension: 1024, lambda: 0.000012
DEBUG:arrowspace.core:Search completed, returning 20 results
INFO:arrowspace.core:Lambda-aware search: k=20
DEBUG:arrowspace.core:Query vector dimension: 1024, lambda: 0.376379
DEBUG:arrowspace.core:Search completed, returning 20 results
INFO:arrowspace.core:Lambda-aware search: k=20
DEBUG:arrowspace.core:Query vector dimension: 1024, lambda: 0.000013
DEBUG:arrowspace.core:Search completed, returning 20 results
INFO:arrowspace.core:Lambda-aware search: k=20
DEBUG:arrowspace.core:Query vector dimension: 1024, lambda: 0.000018
DEBUG:arrowspace.core:Search completed, returning 20 results
INFO:arrowspace.core:Lambda-aware search: k=20
DEBUG:a

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PER-PROFILE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,hit_rate,mean_mrr,mean_ndcg,mean_rec2,mean_gap,n
profile,,,,,,
q_medium,0.815,0.530,0.625,0.144,0.107,200
q_sentence,0.850,0.614,0.690,0.141,0.120,200
q_verbose,0.855,0.639,0.702,0.146,0.117,200


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PER-CATEGORY  (≥3 entries, worst → best NDCG)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Showing 2 categories (filtered from 167 total)


,hit_rate,mean_mrr,mean_ndcg,n_entries
source_category,,,,
debugging,0.556,0.367,0.405,3
image-generation,0.667,0.356,0.535,3


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  HEADLINE  —  pplx-embed-v1-0.6b + ArrowSpace  [subset 200, seed=54]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Hit@10          : 0.840  (504/600)
  Mean MRR       : 0.594
  Mean NDCG@10   : 0.673
  Recall@tier2   : 0.144  (588 entries with tier2 docs)
  Mean score gap : 0.1148
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  WORST 15 ENTRIES  (avg NDCG across profiles)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,entry_id,source_title,source_category,mean_ndcg,mean_mrr,hit_rate,cat_size
0,bq_0104,Career Pivot Assessment,career,0.000,0.000,0.000,12
1,bq_0356,Kubernetes pod troubleshooting guide,debugging,0.000,0.000,0.000,55
2,bq_0317,incident respnse playbok template,cybersecurity,0.000,0.000,0.000,389
3,bq_0301,Automated Ticket Response Template,customer-service,0.000,0.000,0.000,10
4,bq_0431,Email Response Templates for {{situation}},email-efficiency,0.000,0.000,0.000,3
5,bq_0007,protocol question for lab work,academic,0.096,0.033,0.333,27
6,bq_1180,Explain p-values to stakeholders,statistical-analysis,0.100,0.037,0.333,12
7,bq_1374,Structure a Scientific Abstract,writing,0.133,0.042,0.333,44
8,bq_0121,Develop character backstory framework,character-development,0.149,0.042,0.333,9
9,bq_0347,Optimize Slow Database Queries,database,0.216,0.000,0.000,36


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PROFILE ROBUSTNESS  (NDCG delta vs q_medium baseline)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
profile
q_medium      0.0000
q_sentence    0.0647
q_verbose     0.0770

  + = better than q_medium   − = worse than q_medium

Saved → /home/tommaso/code_base/semantic_engine_demo/data/benchmark/eval_pplx06_n200_s54.csv


In [12]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 2 — Search + Evaluation  (fixed)
# ═══════════════════════════════════════════════════════════════════════════

SUBSET_N        = None     # ← None = full benchmark
SUBSET_SEED     = 54
K               = 10
THRESH          = 0.75
MIN_CAT_ENTRIES = 3        # minimum entries per category to show in Report 2

rng_sub  = np.random.default_rng(SUBSET_SEED)
all_idx  = np.arange(len(benchmark))
idx_pool = (rng_sub.choice(all_idx, size=SUBSET_N, replace=False)
            if SUBSET_N and SUBSET_N < len(benchmark)
            else all_idx)

print(f"Running on {len(idx_pool)}/{len(benchmark)} entries  "
      f"(seed={SUBSET_SEED})  ×  {len(PROFILES)} profiles  "
      f"= {len(idx_pool) * len(PROFILES)} searches")

# ── Helpers (both fixed) ───────────────────────────────────────────────────

def _rank_of(results, tier1_set, k):
    """1-based rank of first tier1 hit within top-k only. None if not found."""
    for i, (doc_id, _) in enumerate(results[:k], 1):   # ← capped at k
        if doc_id in tier1_set:
            return i
    return None

def _recall_tier2(results, tier2_set, k):
    """Fraction of retrievable tier2 docs found in top-k.
    Normalises by min(k, |tier2|) — the maximum achievable — not category_size."""
    if not tier2_set:
        return np.nan
    max_retrievable = min(k, len(tier2_set))            # ← correct denominator
    top_k = {doc_id for doc_id, _ in results[:k]}
    return len(top_k & tier2_set) / max_retrievable

# ── Main loop ──────────────────────────────────────────────────────────────
rows = []

for idx in idx_pool:
    entry     = benchmark[idx]
    tier1     = set(entry["relevant_pos"]["tier1"])
    tier2_raw = set(entry["relevant_pos"]["tier2"]) - tier1
    ref       = [(d, 2.0) for d in tier1] + [(d, 1.0) for d in tier2_raw]

    for p_idx, profile in enumerate(PROFILES):
        emb     = queries_emb_all[idx, p_idx]
        results = aspace.search(emb, gl, THRESH)
        scores  = [s for _, s in results]

        rank = _rank_of(results, tier1, K)              # ← K passed explicitly

        rows.append({
            "entry_id"       : entry["entry_id"],
            "source_title"   : entry["source_title"][:45],
            "source_category": entry["source_category"],
            "category_size"  : entry["category_size"],
            "profile"        : profile,
            "query"          : str(entry["queries"].get(profile, ""))[:65],
            "hit"            : rank is not None,        # now truly Hit@K
            "rank"           : rank,
            "mrr"            : 1.0 / rank if rank else 0.0,
            "ndcg"           : float(compute_ndcg(results, ref, k=K)),
            "recall_tier2"   : _recall_tier2(results, tier2_raw, K),
            "top1_score"     : scores[0] if scores else np.nan,
            "score_gap"      : (scores[0] - scores[K-1]) if len(scores) >= K else np.nan,
        })

df      = pd.DataFrame(rows)
n_total = len(df)
SEP     = "━" * 65

# ── REPORT 1: per-profile ──────────────────────────────────────────────────
print(SEP)
print("  PER-PROFILE")
print(SEP)
display(
    df.groupby("profile")
      .agg(hit_rate   = ("hit",          "mean"),
           mean_mrr   = ("mrr",          "mean"),
           mean_ndcg  = ("ndcg",         "mean"),
           mean_rec2  = ("recall_tier2", "mean"),
           mean_gap   = ("score_gap",    "mean"),
           n          = ("entry_id",     "count"))
      .reindex(PROFILES).round(3)
)

# ── REPORT 2: per-category (min entries filter applied) ───────────────────
print(SEP)
print(f"  PER-CATEGORY  (≥{MIN_CAT_ENTRIES} entries, worst → best NDCG)")
print(SEP)
cat_df = (
    df.groupby("source_category")
      .agg(hit_rate  = ("hit",      "mean"),
           mean_mrr  = ("mrr",      "mean"),
           mean_ndcg = ("ndcg",     "mean"),
           n_entries = ("entry_id", lambda x: x.nunique()))
      .round(3)
      .query("n_entries >= @MIN_CAT_ENTRIES")     # ← noise filter
      .sort_values("mean_ndcg")
)
print(f"  Showing {len(cat_df)} categories (filtered from "
      f"{df['source_category'].nunique()} total)")
display(cat_df)

# ── REPORT 3: headline ─────────────────────────────────────────────────────
print(SEP)
print(f"  HEADLINE  —  pplx-embed-v1-0.6b + ArrowSpace  "
      f"[subset {len(idx_pool)}, seed={SUBSET_SEED}]")
print(SEP)
r2 = df["recall_tier2"].dropna()
print(f"  Hit@{K}          : {df['hit'].mean():.3f}  ({int(df['hit'].sum())}/{n_total})")
print(f"  Mean MRR       : {df['mrr'].mean():.3f}")
print(f"  Mean NDCG@{K}   : {df['ndcg'].mean():.3f}")
print(f"  Recall@tier2   : {r2.mean():.3f}  ({len(r2)} entries with tier2 docs)")
print(f"  Mean score gap : {df['score_gap'].mean():.4f}")

# ── REPORT 4: worst 15 entries ────────────────────────────────────────────
print(SEP)
print("  WORST 15 ENTRIES  (avg NDCG across profiles)")
print(SEP)
display(
    df.groupby(["entry_id", "source_title", "source_category"])
      .agg(mean_ndcg = ("ndcg",          "mean"),
           mean_mrr  = ("mrr",           "mean"),
           hit_rate  = ("hit",           "mean"),
           cat_size  = ("category_size", "first"))
      .reset_index()
      .sort_values("mean_ndcg")
      .head(15)
      .reset_index(drop=True)
      .round(3)
)

# ── REPORT 5: profile robustness ──────────────────────────────────────────
print(SEP)
print("  PROFILE ROBUSTNESS  (NDCG delta vs q_medium baseline)")
print(SEP)
pivot  = (df.pivot_table(index="entry_id", columns="profile",
                         values="ndcg", aggfunc="mean")
            .reindex(columns=PROFILES))
deltas = pivot.sub(pivot["q_medium"], axis=0)
print(deltas.mean().round(4).to_string())
print("\n  + = better than q_medium   − = worse than q_medium")

# ── EXPORT ─────────────────────────────────────────────────────────────────
label = f"n{len(idx_pool)}_s{SUBSET_SEED}"
OUT   = DATA_DIR / "benchmark" / f"eval_pplx06_{label}.csv"
df.to_csv(OUT, index=False)
print(f"\nSaved → {OUT}")

INFO:arrowspace.core:Lambda-aware search: k=20
DEBUG:arrowspace.core:Query vector dimension: 1024, lambda: 0.000006


Running on 1379/1379 entries  (seed=54)  ×  3 profiles  = 4137 searches


DEBUG:arrowspace.core:Search completed, returning 20 results
INFO:arrowspace.core:Lambda-aware search: k=20
DEBUG:arrowspace.core:Query vector dimension: 1024, lambda: 0.000007
DEBUG:arrowspace.core:Search completed, returning 20 results
INFO:arrowspace.core:Lambda-aware search: k=20
DEBUG:arrowspace.core:Query vector dimension: 1024, lambda: 0.373923
DEBUG:arrowspace.core:Search completed, returning 20 results
INFO:arrowspace.core:Lambda-aware search: k=20
DEBUG:arrowspace.core:Query vector dimension: 1024, lambda: 0.000008
DEBUG:arrowspace.core:Search completed, returning 20 results
INFO:arrowspace.core:Lambda-aware search: k=20
DEBUG:arrowspace.core:Query vector dimension: 1024, lambda: 0.000016
DEBUG:arrowspace.core:Search completed, returning 20 results
INFO:arrowspace.core:Lambda-aware search: k=20
DEBUG:arrowspace.core:Query vector dimension: 1024, lambda: 0.000013
DEBUG:arrowspace.core:Search completed, returning 20 results
INFO:arrowspace.core:Lambda-aware search: k=20
DEBUG:a

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PER-PROFILE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,hit_rate,mean_mrr,mean_ndcg,mean_rec2,mean_gap,n
profile,,,,,,
q_medium,0.793,0.517,0.611,0.151,0.105,1379
q_sentence,0.828,0.584,0.663,0.158,0.118,1379
q_verbose,0.834,0.619,0.686,0.149,0.116,1379


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PER-CATEGORY  (≥3 entries, worst → best NDCG)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Showing 259 categories (filtered from 377 total)


,hit_rate,mean_mrr,mean_ndcg,n_entries
source_category,,,,
customer-communication,0.467,0.188,0.251,5
hr-operations,0.467,0.130,0.252,5
outreach,0.467,0.141,0.282,5
ticket-response,0.533,0.241,0.299,5
terminology,0.467,0.178,0.313,5
...,...,...,...,...
diversity-initiatives,1.000,0.944,0.938,4
gaming,1.000,0.933,0.946,5
patient-communication,1.000,0.967,0.975,5


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  HEADLINE  —  pplx-embed-v1-0.6b + ArrowSpace  [subset 1379, seed=54]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Hit@10          : 0.818  (3385/4137)
  Mean MRR       : 0.573
  Mean NDCG@10   : 0.654
  Recall@tier2   : 0.152  (4026 entries with tier2 docs)
  Mean score gap : 0.1131
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  WORST 15 ENTRIES  (avg NDCG across profiles)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,entry_id,source_title,source_category,mean_ndcg,mean_mrr,hit_rate,cat_size
0,bq_0081,Develop Brand Voice and Visual Language,brand-identity,0.0,0.0,0.0,20
1,bq_0356,Kubernetes pod troubleshooting guide,debugging,0.0,0.0,0.0,55
2,bq_0385,Write Second Draft Dialogue,dialogue,0.0,0.0,0.0,8
3,bq_0512,Please assist me if you would be so kind,finance,0.0,0.0,0.0,427
4,bq_0942,Product Feedback Categorization,product-feedback,0.0,0.0,0.0,0
5,bq_0709,Looking for some information please,legal,0.0,0.0,0.0,989
6,bq_0933,privacy policy too long,privacy,0.0,0.0,0.0,51
7,bq_1256,Draft Ticket Response Template,templates,0.0,0.0,0.0,10
8,bq_0330,Prepare Dataset for ML Model,data-cleaning,0.0,0.0,0.0,17
9,bq_0104,Career Pivot Assessment,career,0.0,0.0,0.0,12


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PROFILE ROBUSTNESS  (NDCG delta vs q_medium baseline)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
profile
q_medium      0.0000
q_sentence    0.0512
q_verbose     0.0749

  + = better than q_medium   − = worse than q_medium

Saved → /home/tommaso/code_base/semantic_engine_demo/data/benchmark/eval_pplx06_n1379_s54.csv


# Test on multi version of the embeddings

In [1]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 1 — Config + shared setup (run once)
# ═══════════════════════════════════════════════════════════════════════════
import time
import numpy as np
import pandas as pd
from pathlib import Path
from arrowspace import ArrowSpaceBuilder

ROOT     = Path.cwd().parent
DATA_DIR = ROOT / "data"

BENCH_DIR   = DATA_DIR / "benchmark"
BENCH_DIR.mkdir(exist_ok=True)

# ── Benchmark data (loaded once) ───────────────────────────────────────────
import json
with open(BENCH_DIR / "benchmark_queries.json") as f:
    benchmark = json.load(f)

queries_emb_all = np.load(BENCH_DIR / "queries_emb.npy")   # (n_entries, n_profiles, dim)

PROFILES = ["q_medium", "q_sentence", "q_verbose"]         # ← adjust to your actual profile list

# ── Eval config ────────────────────────────────────────────────────────────
SUBSET_N        = 200
SUBSET_SEED     = 54
K               = 10
THRESH          = 0.75
MIN_CAT_ENTRIES = 3

# ── ArrowSpace graph params (same for all variants) ────────────────────────
GRAPH_PARAMS = {
    "eps"  : 1.2,
    "k"    : 25,
    "topk" : 20,
    "p"    : 2.0,
    "sigma": None,
}

# ── Subset index (same sample for all variants → fair comparison) ──────────
rng_sub  = np.random.default_rng(SUBSET_SEED)
all_idx  = np.arange(len(benchmark))
idx_pool = (rng_sub.choice(all_idx, size=SUBSET_N, replace=False)
            if SUBSET_N and SUBSET_N < len(benchmark)
            else all_idx)

print(f"Subset : {len(idx_pool)}/{len(benchmark)} entries  "
      f"(seed={SUBSET_SEED})  ×  {len(PROFILES)} profiles  "
      f"= {len(idx_pool)*len(PROFILES)} searches per variant")

# ── Helpers ────────────────────────────────────────────────────────────────
def _rank_of(results, tier1_set, k):
    for i, (doc_id, _) in enumerate(results[:k], 1):
        if doc_id in tier1_set:
            return i
    return None

def _recall_tier2(results, tier2_set, k):
    if not tier2_set:
        return np.nan
    top_k = {doc_id for doc_id, _ in results[:k]}
    return len(top_k & tier2_set) / min(k, len(tier2_set))

def compute_ndcg(results, ref, k):
    """DCG / IDCG with graded relevance. ref = [(doc_id, gain), ...]"""
    gain_map = {doc_id: g for doc_id, g in ref}
    def dcg(ranking):
        return sum(
            gain_map.get(doc_id, 0) / np.log2(i + 2)
            for i, (doc_id, _) in enumerate(ranking[:k])
        )
    ideal = sorted(ref, key=lambda x: x[1], reverse=True)
    ideal_results = [(d, s) for d, s in ideal]
    idcg = dcg(ideal_results)
    return dcg(results) / idcg if idcg > 0 else 0.0

print("Setup complete.")

Subset : 200/1379 entries  (seed=54)  ×  3 profiles  = 600 searches per variant
Setup complete.


In [2]:
queries_emb_all.shape

(1379, 1024)

In [3]:
v1= np.load(DATA_DIR / "embeddings_v1.npy")

v1.shape

(20000, 1024)

In [4]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 2 — Multi-variant eval loop (fixed)
# ═══════════════════════════════════════════════════════════════════════════

SEP = "━" * 65

VARIANTS = {
    "v1": DATA_DIR / "embeddings_v1.npy",
    "v2": DATA_DIR / "embeddings_v2.npy",
    "v4": DATA_DIR / "embeddings_v4.npy",
    "v5": DATA_DIR / "embeddings_v5.npy",
}

summary_rows = []


def load_query_embs(variant: str) -> np.ndarray:
    """
    Load query embeddings for a given variant.
    Priority:
      1. queries_emb_{variant}.npy  — per-variant re-encoded queries (ideal)
      2. queries_emb.npy            — original single embedding file (fallback)

    Always returns shape (n_entries, n_profiles, dim).
    If the file is 2D (n_entries, dim), all profiles share the same vector.
    """
    per_variant = BENCH_DIR / f"queries_emb_{variant}.npy"
    fallback    = BENCH_DIR / "queries_emb.npy"

    path = per_variant if per_variant.exists() else fallback
    print(f"  Query embeddings: {path.name}")

    raw = np.load(path)
    print(f"  Query emb shape (raw): {raw.shape}")

    if raw.ndim == 3:
        # Already (n_entries, n_profiles, dim) — use directly
        assert raw.shape[0] == len(benchmark), \
            f"Entry count mismatch: {raw.shape[0]} vs {len(benchmark)}"
        return np.ascontiguousarray(raw, dtype=np.float64)

    elif raw.ndim == 2:
        # (n_entries, dim) — no profile axis
        # Expand: every profile gets the same vector
        # ⚠ profiles will be identical until per-variant query files are generated
        expanded = np.stack([raw] * len(PROFILES), axis=1)  # (n, n_profiles, dim)
        return np.ascontiguousarray(expanded, dtype=np.float64)

    else:
        raise ValueError(f"Unexpected query emb shape: {raw.shape}")


for variant, emb_path in VARIANTS.items():

    print(f"\n{SEP}")
    print(f"  VARIANT: {variant}  ({emb_path.name})")
    print(SEP)

    # ── 1. Load corpus embeddings + build index ────────────────────────────
    print(f"  Loading {emb_path.name} ...")
    embs_c = np.ascontiguousarray(np.load(emb_path), dtype=np.float64)
    print(f"  Corpus shape: {embs_c.shape}")

    print("  Building ArrowSpace index...")
    t0 = time.perf_counter()
    aspace, gl = (
        ArrowSpaceBuilder()
        .with_seed(42)
        .with_dims_reduction(enabled=False, eps=None)
        .with_sampling("simple", 1.0)
    ).build_and_store(GRAPH_PARAMS, embs_c)
    print(f"  Build time: {time.perf_counter() - t0:.2f}s")

    # ── 2. Load query embeddings (per-variant or fallback) ─────────────────
    q_embs = load_query_embs(variant)
    # q_embs shape: (n_entries, n_profiles, dim)
    print(f"  Query emb shape (final): {q_embs.shape}")

    profiles_independent = q_embs.shape[1] == len(PROFILES) and \
                           not np.allclose(q_embs[:, 0], q_embs[:, 1])
    if not profiles_independent:
        print("  ⚠ WARNING: all profiles share the same query vector "
              "(run get_query_emb.py to generate per-variant query embeddings)")

    # ── 3. Eval loop ───────────────────────────────────────────────────────
    rows = []
    for idx in idx_pool:
        entry     = benchmark[idx]
        tier1     = set(entry["relevant_pos"]["tier1"])
        tier2_raw = set(entry["relevant_pos"]["tier2"]) - tier1
        ref       = [(d, 2.0) for d in tier1] + [(d, 1.0) for d in tier2_raw]

        for p_idx, profile in enumerate(PROFILES):
            # ✅ fixed: uses p_idx to select the correct profile vector
            emb     = q_embs[idx, p_idx]
            results = aspace.search(emb, gl, THRESH)
            scores  = [s for _, s in results]
            rank    = _rank_of(results, tier1, K)

            rows.append({
                "variant"        : variant,
                "entry_id"       : entry["entry_id"],
                "source_title"   : entry["source_title"][:45],
                "source_category": entry["source_category"],
                "category_size"  : entry["category_size"],
                "profile"        : profile,
                "hit"            : rank is not None,
                "rank"           : rank,
                "mrr"            : 1.0 / rank if rank else 0.0,
                "ndcg"           : float(compute_ndcg(results, ref, k=K)),
                "recall_tier2"   : _recall_tier2(results, tier2_raw, K),
                "top1_score"     : scores[0] if scores else np.nan,
                "score_gap"      : (scores[0] - scores[K-1]) if len(scores) >= K else np.nan,
            })

    df      = pd.DataFrame(rows)
    n_total = len(df)

    # ── 4. Per-profile report ──────────────────────────────────────────────
    print("\n  PER-PROFILE")
    display(
        df.groupby("profile")
          .agg(hit_rate  = ("hit",          "mean"),
               mean_mrr  = ("mrr",          "mean"),
               mean_ndcg = ("ndcg",         "mean"),
               mean_rec2 = ("recall_tier2", "mean"),
               mean_gap  = ("score_gap",    "mean"),
               n         = ("entry_id",     "count"))
          .reindex(PROFILES).round(3)
    )

    # ── 5. Headline ────────────────────────────────────────────────────────
    r2 = df["recall_tier2"].dropna()
    print(f"\n  HEADLINE [{variant}]")
    print(f"  Hit@{K}        : {df['hit'].mean():.3f}  ({int(df['hit'].sum())}/{n_total})")
    print(f"  Mean MRR     : {df['mrr'].mean():.3f}")
    print(f"  Mean NDCG@{K} : {df['ndcg'].mean():.3f}")
    print(f"  Recall@tier2 : {r2.mean():.3f}")
    print(f"  Score gap    : {df['score_gap'].mean():.4f}")

    # ── 6. Export ──────────────────────────────────────────────────────────
    label = f"{variant}_n{len(idx_pool)}_s{SUBSET_SEED}"
    out   = BENCH_DIR / f"eval_{label}.csv"
    df.to_csv(out, index=False)
    print(f"\n  Saved → {out}")

    summary_rows.append({
        "variant"     : variant,
        "hit_rate"    : round(df["hit"].mean(), 3),
        "mean_mrr"    : round(df["mrr"].mean(), 3),
        "mean_ndcg"   : round(df["ndcg"].mean(), 3),
        "recall_tier2": round(r2.mean(), 3),
        "score_gap"   : round(df["score_gap"].mean(), 4),
    })

# ── Final comparison ───────────────────────────────────────────────────────
print(f"\n{SEP}")
print("  VARIANT COMPARISON  (subset={}, seed={})".format(len(idx_pool), SUBSET_SEED))
print(SEP)

comp = pd.DataFrame(summary_rows).set_index("variant")
for col in ["hit_rate", "mean_mrr", "mean_ndcg"]:
    comp[f"Δ{col}"] = (comp[col] - comp.loc["v1", col]).round(3)

display(comp.round(3))
best = comp["mean_ndcg"].idxmax()
print(f"\n  Best NDCG: {best}  ({comp.loc[best,'mean_ndcg']:.3f})")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  VARIANT: v1  (embeddings_v1.npy)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Loading embeddings_v1.npy ...
  Corpus shape: (20000, 1024)
  Building ArrowSpace index...
  Build time: 160.26s
  Query embeddings: queries_emb.npy
  Query emb shape (raw): (1379, 1024)
  Query emb shape (final): (1379, 3, 1024)
  ⚠ WARNING: all profiles share the same query vector (run get_query_emb.py to generate per-variant query embeddings)

  PER-PROFILE


,hit_rate,mean_mrr,mean_ndcg,mean_rec2,mean_gap,n
profile,,,,,,
q_medium,0.73,0.509,0.36,0.152,0.116,200
q_sentence,0.73,0.509,0.36,0.152,0.116,200
q_verbose,0.73,0.509,0.36,0.152,0.116,200



  HEADLINE [v1]
  Hit@10        : 0.730  (438/600)
  Mean MRR     : 0.509
  Mean NDCG@10 : 0.360
  Recall@tier2 : 0.152
  Score gap    : 0.1159

  Saved → /home/tommaso/code_base/semantic_engine_demo/data/benchmark/eval_v1_n200_s54.csv

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  VARIANT: v2  (embeddings_v2.npy)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Loading embeddings_v2.npy ...
  Corpus shape: (20000, 1024)
  Building ArrowSpace index...
  Build time: 119.80s
  Query embeddings: queries_emb.npy
  Query emb shape (raw): (1379, 1024)
  Query emb shape (final): (1379, 3, 1024)
  ⚠ WARNING: all profiles share the same query vector (run get_query_emb.py to generate per-variant query embeddings)

  PER-PROFILE


,hit_rate,mean_mrr,mean_ndcg,mean_rec2,mean_gap,n
profile,,,,,,
q_medium,0.735,0.518,0.361,0.16,0.137,200
q_sentence,0.735,0.518,0.361,0.16,0.137,200
q_verbose,0.735,0.518,0.361,0.16,0.137,200



  HEADLINE [v2]
  Hit@10        : 0.735  (441/600)
  Mean MRR     : 0.518
  Mean NDCG@10 : 0.361
  Recall@tier2 : 0.160
  Score gap    : 0.1366

  Saved → /home/tommaso/code_base/semantic_engine_demo/data/benchmark/eval_v2_n200_s54.csv

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  VARIANT: v4  (embeddings_v4.npy)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Loading embeddings_v4.npy ...
  Corpus shape: (20000, 1024)
  Building ArrowSpace index...
  Build time: 160.62s
  Query embeddings: queries_emb.npy
  Query emb shape (raw): (1379, 1024)
  Query emb shape (final): (1379, 3, 1024)
  ⚠ WARNING: all profiles share the same query vector (run get_query_emb.py to generate per-variant query embeddings)

  PER-PROFILE


...


  HEADLINE [v4]
  Hit@10        : 0.700  (420/600)
  Mean MRR     : 0.476
  Mean NDCG@10 : 0.345
  Recall@tier2 : 0.166
  Score gap    : 0.1258

  Saved → /home/tommaso/code_base/semantic_engine_demo/data/benchmark/eval_v4_n200_s54.csv

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  VARIANT: v5  (embeddings_v5.npy)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Loading embeddings_v5.npy ...
  Corpus shape: (20000, 1024)
  Building ArrowSpace index...
  Build time: 140.93s
  Query embeddings: queries_emb.npy
  Query emb shape (raw): (1379, 1024)
  Query emb shape (final): (1379, 3, 1024)
  ⚠ WARNING: all profiles share the same query vector (run get_query_emb.py to generate per-variant query embeddings)

  PER-PROFILE


,hit_rate,mean_mrr,mean_ndcg,mean_rec2,mean_gap,n
profile,,,,,,
q_medium,0.69,0.477,0.345,0.166,0.118,200
q_sentence,0.69,0.477,0.345,0.166,0.118,200
q_verbose,0.69,0.477,0.345,0.166,0.118,200



  HEADLINE [v5]
  Hit@10        : 0.690  (414/600)
  Mean MRR     : 0.477
  Mean NDCG@10 : 0.345
  Recall@tier2 : 0.166
  Score gap    : 0.1178

  Saved → /home/tommaso/code_base/semantic_engine_demo/data/benchmark/eval_v5_n200_s54.csv

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  VARIANT COMPARISON  (subset=200, seed=54)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,hit_rate,mean_mrr,mean_ndcg,recall_tier2,score_gap,Δhit_rate,Δmean_mrr,Δmean_ndcg
variant,,,,,,,,
v1,0.730,0.509,0.360,0.152,0.116,0.000,0.000,0.000
v2,0.735,0.518,0.361,0.160,0.137,0.005,0.009,0.001
v4,0.700,0.476,0.345,0.166,0.126,-0.030,-0.033,-0.015
v5,0.690,0.477,0.345,0.166,0.118,-0.040,-0.032,-0.015



  Best NDCG: v2  (0.361)


In [5]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 2 — Multi-variant eval loop (fixed)
# ═══════════════════════════════════════════════════════════════════════════

SEP = "━" * 65

VARIANTS = {
    "v1": DATA_DIR / "embeddings_v1.npy",
    "v2": DATA_DIR / "embeddings_v2.npy",
    "v4": DATA_DIR / "embeddings_v4.npy",
    "v5": DATA_DIR / "embeddings_v5.npy",
}

summary_rows = []


def load_query_embs(variant: str) -> np.ndarray:
    """
    Load query embeddings for a given variant.
    Priority:
      1. queries_emb_{variant}.npy  — per-variant re-encoded queries (ideal)
      2. queries_emb.npy            — original single embedding file (fallback)

    Always returns shape (n_entries, n_profiles, dim).
    If the file is 2D (n_entries, dim), all profiles share the same vector.
    """
    per_variant = BENCH_DIR / f"queries_emb_all.npy"
    fallback    = BENCH_DIR / "queries_emb.npy"

    path = per_variant if per_variant.exists() else fallback
    print(f"  Query embeddings: {path.name}")

    raw = np.load(path)
    print(f"  Query emb shape (raw): {raw.shape}")

    if raw.ndim == 3:
        # Already (n_entries, n_profiles, dim) — use directly
        assert raw.shape[0] == len(benchmark), \
            f"Entry count mismatch: {raw.shape[0]} vs {len(benchmark)}"
        return np.ascontiguousarray(raw, dtype=np.float64)

    elif raw.ndim == 2:
        # (n_entries, dim) — no profile axis
        # Expand: every profile gets the same vector
        # ⚠ profiles will be identical until per-variant query files are generated
        expanded = np.stack([raw] * len(PROFILES), axis=1)  # (n, n_profiles, dim)
        return np.ascontiguousarray(expanded, dtype=np.float64)

    else:
        raise ValueError(f"Unexpected query emb shape: {raw.shape}")


for variant, emb_path in VARIANTS.items():

    print(f"\n{SEP}")
    print(f"  VARIANT: {variant}  ({emb_path.name})")
    print(SEP)

    # ── 1. Load corpus embeddings + build index ────────────────────────────
    print(f"  Loading {emb_path.name} ...")
    embs_c = np.ascontiguousarray(np.load(emb_path), dtype=np.float64)
    print(f"  Corpus shape: {embs_c.shape}")

    print("  Building ArrowSpace index...")
    t0 = time.perf_counter()
    aspace, gl = (
        ArrowSpaceBuilder()
        .with_seed(42)
        .with_dims_reduction(enabled=False, eps=None)
        .with_sampling("simple", 1.0)
    ).build_and_store(GRAPH_PARAMS, embs_c)
    print(f"  Build time: {time.perf_counter() - t0:.2f}s")

    # ── 2. Load query embeddings (per-variant or fallback) ─────────────────
    q_embs = load_query_embs(variant)
    # q_embs shape: (n_entries, n_profiles, dim)
    print(f"  Query emb shape (final): {q_embs.shape}")

    profiles_independent = q_embs.shape[1] == len(PROFILES) and \
                           not np.allclose(q_embs[:, 0], q_embs[:, 1])
    if not profiles_independent:
        print("  ⚠ WARNING: all profiles share the same query vector "
              "(run get_query_emb.py to generate per-variant query embeddings)")

    # ── 3. Eval loop ───────────────────────────────────────────────────────
    rows = []
    for idx in idx_pool:
        entry     = benchmark[idx]
        tier1     = set(entry["relevant_pos"]["tier1"])
        tier2_raw = set(entry["relevant_pos"]["tier2"]) - tier1
        ref       = [(d, 2.0) for d in tier1] + [(d, 1.0) for d in tier2_raw]

        for p_idx, profile in enumerate(PROFILES):
            # ✅ fixed: uses p_idx to select the correct profile vector
            emb     = q_embs[idx, p_idx]
            results = aspace.search(emb, gl, THRESH)
            scores  = [s for _, s in results]
            rank    = _rank_of(results, tier1, K)

            rows.append({
                "variant"        : variant,
                "entry_id"       : entry["entry_id"],
                "source_title"   : entry["source_title"][:45],
                "source_category": entry["source_category"],
                "category_size"  : entry["category_size"],
                "profile"        : profile,
                "hit"            : rank is not None,
                "rank"           : rank,
                "mrr"            : 1.0 / rank if rank else 0.0,
                "ndcg"           : float(compute_ndcg(results, ref, k=K)),
                "recall_tier2"   : _recall_tier2(results, tier2_raw, K),
                "top1_score"     : scores[0] if scores else np.nan,
                "score_gap"      : (scores[0] - scores[K-1]) if len(scores) >= K else np.nan,
            })

    df      = pd.DataFrame(rows)
    n_total = len(df)

    # ── 4. Per-profile report ──────────────────────────────────────────────
    print("\n  PER-PROFILE")
    display(
        df.groupby("profile")
          .agg(hit_rate  = ("hit",          "mean"),
               mean_mrr  = ("mrr",          "mean"),
               mean_ndcg = ("ndcg",         "mean"),
               mean_rec2 = ("recall_tier2", "mean"),
               mean_gap  = ("score_gap",    "mean"),
               n         = ("entry_id",     "count"))
          .reindex(PROFILES).round(3)
    )

    # ── 5. Headline ────────────────────────────────────────────────────────
    r2 = df["recall_tier2"].dropna()
    print(f"\n  HEADLINE [{variant}]")
    print(f"  Hit@{K}        : {df['hit'].mean():.3f}  ({int(df['hit'].sum())}/{n_total})")
    print(f"  Mean MRR     : {df['mrr'].mean():.3f}")
    print(f"  Mean NDCG@{K} : {df['ndcg'].mean():.3f}")
    print(f"  Recall@tier2 : {r2.mean():.3f}")
    print(f"  Score gap    : {df['score_gap'].mean():.4f}")

    # ── 6. Export ──────────────────────────────────────────────────────────
    label = f"{variant}_n{len(idx_pool)}_s{SUBSET_SEED}"
    out   = BENCH_DIR / f"eval_{label}.csv"
    df.to_csv(out, index=False)
    print(f"\n  Saved → {out}")

    summary_rows.append({
        "variant"     : variant,
        "hit_rate"    : round(df["hit"].mean(), 3),
        "mean_mrr"    : round(df["mrr"].mean(), 3),
        "mean_ndcg"   : round(df["ndcg"].mean(), 3),
        "recall_tier2": round(r2.mean(), 3),
        "score_gap"   : round(df["score_gap"].mean(), 4),
    })

# ── Final comparison ───────────────────────────────────────────────────────
print(f"\n{SEP}")
print("  VARIANT COMPARISON  (subset={}, seed={})".format(len(idx_pool), SUBSET_SEED))
print(SEP)

comp = pd.DataFrame(summary_rows).set_index("variant")
for col in ["hit_rate", "mean_mrr", "mean_ndcg"]:
    comp[f"Δ{col}"] = (comp[col] - comp.loc["v1", col]).round(3)

display(comp.round(3))
best = comp["mean_ndcg"].idxmax()
print(f"\n  Best NDCG: {best}  ({comp.loc[best,'mean_ndcg']:.3f})")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  VARIANT: v1  (embeddings_v1.npy)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Loading embeddings_v1.npy ...
  Corpus shape: (20000, 1024)
  Building ArrowSpace index...
  Build time: 196.00s
  Query embeddings: queries_emb_all.npy
  Query emb shape (raw): (1379, 3, 1024)
  Query emb shape (final): (1379, 3, 1024)

  PER-PROFILE


,hit_rate,mean_mrr,mean_ndcg,mean_rec2,mean_gap,n
profile,,,,,,
q_medium,0.710,0.460,0.338,0.148,0.107,200
q_sentence,0.725,0.512,0.368,0.152,0.118,200
q_verbose,0.735,0.561,0.374,0.160,0.116,200



  HEADLINE [v1]
  Hit@10        : 0.723  (434/600)
  Mean MRR     : 0.511
  Mean NDCG@10 : 0.360
  Recall@tier2 : 0.154
  Score gap    : 0.1137

  Saved → /home/tommaso/code_base/semantic_engine_demo/data/benchmark/eval_v1_n200_s54.csv

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  VARIANT: v2  (embeddings_v2.npy)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Loading embeddings_v2.npy ...
  Corpus shape: (20000, 1024)
  Building ArrowSpace index...
  Build time: 113.48s
  Query embeddings: queries_emb_all.npy
  Query emb shape (raw): (1379, 3, 1024)
  Query emb shape (final): (1379, 3, 1024)

  PER-PROFILE


,hit_rate,mean_mrr,mean_ndcg,mean_rec2,mean_gap,n
profile,,,,,,
q_medium,0.760,0.513,0.379,0.171,0.135,200
q_sentence,0.740,0.556,0.387,0.160,0.140,200
q_verbose,0.715,0.504,0.364,0.164,0.128,200



  HEADLINE [v2]
  Hit@10        : 0.738  (443/600)
  Mean MRR     : 0.524
  Mean NDCG@10 : 0.377
  Recall@tier2 : 0.165
  Score gap    : 0.1341

  Saved → /home/tommaso/code_base/semantic_engine_demo/data/benchmark/eval_v2_n200_s54.csv

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  VARIANT: v4  (embeddings_v4.npy)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Loading embeddings_v4.npy ...
  Corpus shape: (20000, 1024)
  Building ArrowSpace index...
  Build time: 124.84s
  Query embeddings: queries_emb_all.npy
  Query emb shape (raw): (1379, 3, 1024)
  Query emb shape (final): (1379, 3, 1024)

  PER-PROFILE


...


  HEADLINE [v4]
  Hit@10        : 0.703  (422/600)
  Mean MRR     : 0.494
  Mean NDCG@10 : 0.360
  Recall@tier2 : 0.169
  Score gap    : 0.1239

  Saved → /home/tommaso/code_base/semantic_engine_demo/data/benchmark/eval_v4_n200_s54.csv

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  VARIANT: v5  (embeddings_v5.npy)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Loading embeddings_v5.npy ...
  Corpus shape: (20000, 1024)
  Building ArrowSpace index...
  Build time: 113.98s
  Query embeddings: queries_emb_all.npy
  Query emb shape (raw): (1379, 3, 1024)
  Query emb shape (final): (1379, 3, 1024)

  PER-PROFILE


,hit_rate,mean_mrr,mean_ndcg,mean_rec2,mean_gap,n
profile,,,,,,
q_medium,0.70,0.476,0.350,0.163,0.111,200
q_sentence,0.72,0.500,0.363,0.161,0.121,200
q_verbose,0.68,0.498,0.352,0.158,0.118,200



  HEADLINE [v5]
  Hit@10        : 0.700  (420/600)
  Mean MRR     : 0.491
  Mean NDCG@10 : 0.355
  Recall@tier2 : 0.161
  Score gap    : 0.1166

  Saved → /home/tommaso/code_base/semantic_engine_demo/data/benchmark/eval_v5_n200_s54.csv

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  VARIANT COMPARISON  (subset=200, seed=54)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,hit_rate,mean_mrr,mean_ndcg,recall_tier2,score_gap,Δhit_rate,Δmean_mrr,Δmean_ndcg
variant,,,,,,,,
v1,0.723,0.511,0.360,0.154,0.114,0.000,0.000,0.000
v2,0.738,0.524,0.377,0.165,0.134,0.015,0.013,0.017
v4,0.703,0.494,0.360,0.169,0.124,-0.020,-0.017,0.000
v5,0.700,0.491,0.355,0.161,0.117,-0.023,-0.020,-0.005



  Best NDCG: v2  (0.377)


## Run full evaluation only for version 2:

In [6]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 1 — Config + shared setup (run once)
# ═══════════════════════════════════════════════════════════════════════════
import time
import numpy as np
import pandas as pd
from pathlib import Path
from arrowspace import ArrowSpaceBuilder

ROOT     = Path.cwd().parent
DATA_DIR = ROOT / "data"

BENCH_DIR   = DATA_DIR / "benchmark"
BENCH_DIR.mkdir(exist_ok=True)

# ── Benchmark data (loaded once) ───────────────────────────────────────────
import json
with open(BENCH_DIR / "benchmark_queries.json") as f:
    benchmark = json.load(f)

queries_emb_all = np.load(BENCH_DIR / "queries_emb.npy")   # (n_entries, n_profiles, dim)

PROFILES = ["q_medium", "q_sentence", "q_verbose"]         # ← adjust to your actual profile list

# ── Eval config ────────────────────────────────────────────────────────────
SUBSET_N        = None
SUBSET_SEED     = 54
K               = 10
THRESH          = 0.75
MIN_CAT_ENTRIES = 3

# ── ArrowSpace graph params (same for all variants) ────────────────────────
GRAPH_PARAMS = {
    "eps"  : 1.2,
    "k"    : 25,
    "topk" : 20,
    "p"    : 2.0,
    "sigma": None,
}

# ── Subset index (same sample for all variants → fair comparison) ──────────
rng_sub  = np.random.default_rng(SUBSET_SEED)
all_idx  = np.arange(len(benchmark))
idx_pool = (rng_sub.choice(all_idx, size=SUBSET_N, replace=False)
            if SUBSET_N and SUBSET_N < len(benchmark)
            else all_idx)

print(f"Subset : {len(idx_pool)}/{len(benchmark)} entries  "
      f"(seed={SUBSET_SEED})  ×  {len(PROFILES)} profiles  "
      f"= {len(idx_pool)*len(PROFILES)} searches per variant")

# ── Helpers ────────────────────────────────────────────────────────────────
def _rank_of(results, tier1_set, k):
    for i, (doc_id, _) in enumerate(results[:k], 1):
        if doc_id in tier1_set:
            return i
    return None

def _recall_tier2(results, tier2_set, k):
    if not tier2_set:
        return np.nan
    top_k = {doc_id for doc_id, _ in results[:k]}
    return len(top_k & tier2_set) / min(k, len(tier2_set))

def compute_ndcg(results, ref, k):
    """DCG / IDCG with graded relevance. ref = [(doc_id, gain), ...]"""
    gain_map = {doc_id: g for doc_id, g in ref}
    def dcg(ranking):
        return sum(
            gain_map.get(doc_id, 0) / np.log2(i + 2)
            for i, (doc_id, _) in enumerate(ranking[:k])
        )
    ideal = sorted(ref, key=lambda x: x[1], reverse=True)
    ideal_results = [(d, s) for d, s in ideal]
    idcg = dcg(ideal_results)
    return dcg(results) / idcg if idcg > 0 else 0.0

print("Setup complete.")

Subset : 1379/1379 entries  (seed=54)  ×  3 profiles  = 4137 searches per variant
Setup complete.


In [7]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 2 — Multi-variant eval loop (fixed)
# ═══════════════════════════════════════════════════════════════════════════

SEP = "━" * 65

VARIANTS = {
    "v2": DATA_DIR / "embeddings_v2.npy",
}

summary_rows = []


def load_query_embs(variant: str) -> np.ndarray:
    """
    Load query embeddings for a given variant.
    Priority:
      1. queries_emb_{variant}.npy  — per-variant re-encoded queries (ideal)
      2. queries_emb.npy            — original single embedding file (fallback)

    Always returns shape (n_entries, n_profiles, dim).
    If the file is 2D (n_entries, dim), all profiles share the same vector.
    """
    per_variant = BENCH_DIR / f"queries_emb_all.npy"
    fallback    = BENCH_DIR / "queries_emb.npy"

    path = per_variant if per_variant.exists() else fallback
    print(f"  Query embeddings: {path.name}")

    raw = np.load(path)
    print(f"  Query emb shape (raw): {raw.shape}")

    if raw.ndim == 3:
        # Already (n_entries, n_profiles, dim) — use directly
        assert raw.shape[0] == len(benchmark), \
            f"Entry count mismatch: {raw.shape[0]} vs {len(benchmark)}"
        return np.ascontiguousarray(raw, dtype=np.float64)

    elif raw.ndim == 2:
        # (n_entries, dim) — no profile axis
        # Expand: every profile gets the same vector
        # ⚠ profiles will be identical until per-variant query files are generated
        expanded = np.stack([raw] * len(PROFILES), axis=1)  # (n, n_profiles, dim)
        return np.ascontiguousarray(expanded, dtype=np.float64)

    else:
        raise ValueError(f"Unexpected query emb shape: {raw.shape}")


for variant, emb_path in VARIANTS.items():

    print(f"\n{SEP}")
    print(f"  VARIANT: {variant}  ({emb_path.name})")
    print(SEP)

    # ── 1. Load corpus embeddings + build index ────────────────────────────
    print(f"  Loading {emb_path.name} ...")
    embs_c = np.ascontiguousarray(np.load(emb_path), dtype=np.float64)
    print(f"  Corpus shape: {embs_c.shape}")

    print("  Building ArrowSpace index...")
    t0 = time.perf_counter()
    aspace, gl = (
        ArrowSpaceBuilder()
        .with_seed(42)
        .with_dims_reduction(enabled=False, eps=None)
        .with_sampling("simple", 1.0)
    ).build_and_store(GRAPH_PARAMS, embs_c)
    print(f"  Build time: {time.perf_counter() - t0:.2f}s")

    # ── 2. Load query embeddings (per-variant or fallback) ─────────────────
    q_embs = load_query_embs(variant)
    # q_embs shape: (n_entries, n_profiles, dim)
    print(f"  Query emb shape (final): {q_embs.shape}")

    profiles_independent = q_embs.shape[1] == len(PROFILES) and \
                           not np.allclose(q_embs[:, 0], q_embs[:, 1])
    if not profiles_independent:
        print("  ⚠ WARNING: all profiles share the same query vector "
              "(run get_query_emb.py to generate per-variant query embeddings)")

    # ── 3. Eval loop ───────────────────────────────────────────────────────
    rows = []
    for idx in idx_pool:
        entry     = benchmark[idx]
        tier1     = set(entry["relevant_pos"]["tier1"])
        tier2_raw = set(entry["relevant_pos"]["tier2"]) - tier1
        ref       = [(d, 2.0) for d in tier1] + [(d, 1.0) for d in tier2_raw]

        for p_idx, profile in enumerate(PROFILES):
            emb     = q_embs[idx, p_idx]
            results = aspace.search(emb, gl, THRESH)
            scores  = [s for _, s in results]
            rank    = _rank_of(results, tier1, K)

            rows.append({
                "variant"        : variant,
                "entry_id"       : entry["entry_id"],
                "source_title"   : entry["source_title"][:45],
                "source_category": entry["source_category"],
                "category_size"  : entry["category_size"],
                "profile"        : profile,
                "hit"            : rank is not None,
                "rank"           : rank,
                "mrr"            : 1.0 / rank if rank else 0.0,
                "ndcg"           : float(compute_ndcg(results, ref, k=K)),
                "recall_tier2"   : _recall_tier2(results, tier2_raw, K),
                "top1_score"     : scores[0] if scores else np.nan,
                "score_gap"      : (scores[0] - scores[K-1]) if len(scores) >= K else np.nan,
            })

    df      = pd.DataFrame(rows)
    n_total = len(df)

    # ── 4. Per-profile report
    print("\n  PER-PROFILE")
    display(
        df.groupby("profile")
          .agg(hit_rate  = ("hit",          "mean"),
               mean_mrr  = ("mrr",          "mean"),
               mean_ndcg = ("ndcg",         "mean"),
               mean_rec2 = ("recall_tier2", "mean"),
               mean_gap  = ("score_gap",    "mean"),
               n         = ("entry_id",     "count"))
          .reindex(PROFILES).round(3)
    )

    # ── 5. Headline ────────────────────────────────────────────────────────
    r2 = df["recall_tier2"].dropna()
    print(f"\n  HEADLINE [{variant}]")
    print(f"  Hit@{K}        : {df['hit'].mean():.3f}  ({int(df['hit'].sum())}/{n_total})")
    print(f"  Mean MRR     : {df['mrr'].mean():.3f}")
    print(f"  Mean NDCG@{K} : {df['ndcg'].mean():.3f}")
    print(f"  Recall@tier2 : {r2.mean():.3f}")
    print(f"  Score gap    : {df['score_gap'].mean():.4f}")

    # ── 6. Export ──────────────────────────────────────────────────────────
    label = f"{variant}_n{len(idx_pool)}_s{SUBSET_SEED}"
    out   = BENCH_DIR / f"eval_{label}.csv"
    df.to_csv(out, index=False)
    print(f"\n  Saved → {out}")

    summary_rows.append({
        "variant"     : variant,
        "hit_rate"    : round(df["hit"].mean(), 3),
        "mean_mrr"    : round(df["mrr"].mean(), 3),
        "mean_ndcg"   : round(df["ndcg"].mean(), 3),
        "recall_tier2": round(r2.mean(), 3),
        "score_gap"   : round(df["score_gap"].mean(), 4),
    })

# ── Final comparison ───────────────────────────────────────────────────────
print(f"\n{SEP}")
print("  VARIANT COMPARISON  (subset={}, seed={})".format(len(idx_pool), SUBSET_SEED))
print(SEP)

comp = pd.DataFrame(summary_rows).set_index("variant")
for col in ["hit_rate", "mean_mrr", "mean_ndcg"]:
    comp[f"Δ{col}"] = (comp[col] - comp.loc["v1", col]).round(3)

display(comp.round(3))
best = comp["mean_ndcg"].idxmax()
print(f"\n  Best NDCG: {best}  ({comp.loc[best,'mean_ndcg']:.3f})")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  VARIANT: v2  (embeddings_v2.npy)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Loading embeddings_v2.npy ...
  Corpus shape: (20000, 1024)
  Building ArrowSpace index...
  Build time: 144.74s
  Query embeddings: queries_emb_all.npy
  Query emb shape (raw): (1379, 3, 1024)
  Query emb shape (final): (1379, 3, 1024)

  PER-PROFILE


,hit_rate,mean_mrr,mean_ndcg,mean_rec2,mean_gap,n
profile,,,,,,
q_medium,0.743,0.505,0.372,0.174,0.131,1379
q_sentence,0.751,0.531,0.384,0.175,0.135,1379
q_verbose,0.730,0.520,0.372,0.166,0.125,1379



  HEADLINE [v2]
  Hit@10        : 0.741  (3066/4137)
  Mean MRR     : 0.518
  Mean NDCG@10 : 0.376
  Recall@tier2 : 0.172
  Score gap    : 0.1302

  Saved → /home/tommaso/code_base/semantic_engine_demo/data/benchmark/eval_v2_n1379_s54.csv

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  VARIANT COMPARISON  (subset=1379, seed=54)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


KeyError: 'v1'

# Nomic embeddings full tests

In [1]:
import json
import time
import warnings
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from arrowspace import ArrowSpaceBuilder

warnings.filterwarnings("ignore")

# ── PATHS ─────────────────────────────────────────────────────────────────────
ROOT        = Path("/home/tommaso/code_base/semantic_engine_demo")
NOMIC_DIR   = ROOT / "data" / "nomic_emb"         # corpus .npy files live here
BENCH_FILE  = ROOT / "data" / "benchmark" / "benchmark_queries_01.json"
QUERY_DIR   = ROOT / "data" / "nomic_bench"                          # query .npy files will go here too
OUT_DIR     = NOMIC_DIR                            # eval CSVs land here

EXTRACTIONS = ["compact", "full", "structured"]
DIMS        = [768, 512, 256]
PROFILES    = ["q_medium", "q_sentence", "q_verbose"]
TOP_K        = 10

print("Corpus dir :", NOMIC_DIR)
print("Bench file :", BENCH_FILE)
print("Query dir  :", QUERY_DIR)

Corpus dir : /home/tommaso/code_base/semantic_engine_demo/data/nomic_emb
Bench file : /home/tommaso/code_base/semantic_engine_demo/data/benchmark/benchmark_queries_01.json
Query dir  : /home/tommaso/code_base/semantic_engine_demo/data/nomic_bench


In [2]:
with open(BENCH_FILE) as f:
    BQ: list[dict] = json.load(f)

N_QUERIES = len(BQ)
print(f"Benchmark entries : {N_QUERIES}")
print(f"Query fields      : {list(BQ[0]['queries'].keys())}")
print(f"tier2 ∋ tier1?    : {BQ[0]['relevant_pos']['tier1'][0] in BQ[0]['relevant_pos']['tier2']}")
print()
print("Sample entry:")
print(f"  entry_id   : {BQ[0]['entry_id']}")
print(f"  source_pos : {BQ[0]['source_pos']}")
print(f"  q_medium   : {BQ[0]['queries']['q_medium']!r}")
print(f"  q_sentence : {BQ[0]['queries']['q_sentence']!r}")
print(f"  q_verbose  : {BQ[0]['queries']['q_verbose']!r}")
print(f"  tier1      : {BQ[0]['relevant_pos']['tier1']}")
print(f"  tier2      : {BQ[0]['relevant_pos']['tier2'][:5]} ...")

Benchmark entries : 1379
Query fields      : ['q_medium', 'q_sentence', 'q_verbose']
tier2 ∋ tier1?    : False

Sample entry:
  entry_id   : bq_0000
  source_pos : 14341
  q_medium   : 'A/B test analysis for product flow optimization'
  q_sentence : 'Looking for insights on reducing onboarding steps via A/B testing'
  q_verbose  : 'Need detailed results on changing onboarding from 5 to 3 steps for {{product_name}}'
  tier1      : [14341]
  tier2      : [2097, 9418, 10829, 13080, 13600] ...


In [3]:
def score_single(
    results: list[tuple[int, float]],
    tier1: list[int],
    tier2: list[int],
    k: int = TOP_K,
) -> dict:
    """
    Compute all retrieval metrics for one (query, index) pair.

    tier2 in benchmark_queries_01.json does NOT include the tier1 doc.
    Relevance weights: tier1 doc = 2,  tier2 doc = 1,  other = 0.
    recall_tier2 denominator = len(tier2)  (excludes tier1).
    """
    ranked      = [r[0] for r in results[:k]]
    tier1_set   = set(tier1)
    tier2_set   = set(tier2)

    # ── hit@k & MRR (tier-1 strict) ──────────────────────────────────────────
    rank = None
    for i, doc_pos in enumerate(ranked):
        if doc_pos in tier1_set:
            rank = i + 1
            break
    hit = rank is not None
    mrr = 1.0 / rank if rank else 0.0

    # ── nDCG@k (graded: tier1=2, tier2=1) ────────────────────────────────────
    gains = [
        2 if p in tier1_set else (1 if p in tier2_set else 0)
        for p in ranked
    ]
    dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains))

    # ideal: one tier1 doc + as many tier2 as fit in top-k
    ideal_gains = sorted(
        [2] * len(tier1) + [1] * len(tier2), reverse=True
    )[:k]
    ideal_dcg = sum(g / np.log2(i + 2) for i, g in enumerate(ideal_gains))
    ndcg = dcg / ideal_dcg if ideal_dcg > 0 else 0.0

    # ── recall@k tier-2 ───────────────────────────────────────────────────────
    # denominator = len(tier2) because tier2 list excludes tier1 in this benchmark
    t2_found = len(tier2_set.intersection(set(ranked)))
    rec2     = t2_found / len(tier2_set) if len(tier2_set) > 0 else 0.0

    # ── score gap  (top-1 score − last-in-results score) ─────────────────────
    top1_score = results[0][1] if results else 0.0
    score_gap  = (results[0][1] - results[-1][1]) if len(results) > 1 else 0.0

    return dict(
        hit          = hit,
        rank         = float(rank) if rank else float("nan"),
        mrr          = mrr,
        ndcg         = ndcg,
        recall_tier2 = rec2,
        top1_score   = top1_score,
        score_gap    = score_gap,
    )


print("Metrics helper defined.")
print("  Metrics: hit, rank, mrr, ndcg, recall_tier2, top1_score, score_gap")

Metrics helper defined.
  Metrics: hit, rank, mrr, ndcg, recall_tier2, top1_score, score_gap


In [4]:
from arrowspace import ArrowSpaceBuilder

# ── pattern completo dei messaggi λ=0 di ArrowSpace ──────────────────────────
_LAMBDA_ZERO_MSGS = (
    "lambda is zero",     # ValueError: Lambda is zero - check item magnitude...
    "lambda is 0",        # PanicException: Lambda is zero - check...
    "check your eps parameter",
    "undecidable",
    "item magnitude",
)

def _is_lambda_zero(e: Exception) -> bool:
    msg = str(e).lower()
    return any(pat in msg for pat in _LAMBDA_ZERO_MSGS)


def get_graph_params(d: int) -> dict:
    """
    Dimension-aware ArrowSpace wiring.

        768d: eps = 0.75   (leggermente più stretto del default 0.80,
                            per gestire varianti full con testo lungo)
        512d: eps = 0.55
        256d: eps = 0.35
    """
    if d >= 768:
        eps = 0.75
    elif d >= 512:
        eps = 0.55
    else:
        eps = 0.35

    return {"eps": eps, "k": 25, "topk": 12, "p": 2.0, "sigma": None}


def build_arrowspace(corpus_vecs: np.ndarray):
    """Build ArrowSpace index. Richiede float64 in input."""
    embs   = corpus_vecs.astype(np.float64)
    d      = embs.shape[1]
    params = get_graph_params(d)
    print(f"  Graph params : {params}")
    aspace, gl = (
        ArrowSpaceBuilder()
        .with_seed(42)
        .with_dims_reduction(enabled=False, eps=None)
        .with_sampling("simple", 1.0)
    ).build_and_store(params, embs)
    return aspace, gl


def safe_search(aspace, gl, qvec: np.ndarray, alpha: float = 0.75):
    """
    Wrapper su aspace.search.
    Restituisce [] per qualsiasi variante di λ=0 invece di crashare.
    """
    try:
        return aspace.search(qvec, gl, alpha)
    except Exception as e:
        if _is_lambda_zero(e):
            return []
        raise


def evaluate_variant(
    label: str,
    corpus_path: Path,
    query_path: Path,
    bq: list[dict],
) -> pd.DataFrame:

    print(f"\n{'━' * 60}")
    print(f"  {label}")
    print(f"{'━' * 60}")

    corpus_vecs = np.load(corpus_path)
    q_vecs      = np.load(query_path)
    d = corpus_vecs.shape[1]

    print(f"  Corpus shape : {corpus_vecs.shape}")
    print(f"  Query shape  : {q_vecs.shape}")

    assert q_vecs.shape[0] == len(bq), "Query count mismatch"
    assert q_vecs.shape[1] == 3,       "Expected 3 profiles on axis 1"
    assert q_vecs.shape[2] == d,       "Dim mismatch between corpus and queries"

    print("  Building ArrowSpace index...", flush=True)
    t0         = time.perf_counter()
    aspace, gl = build_arrowspace(corpus_vecs)
    build_t    = time.perf_counter() - t0
    print(f"  Build time   : {build_t:.2f}s")

    rows = []
    lambda_zero_count = 0

    for qi, entry in enumerate(bq):
        tier1 = entry["relevant_pos"]["tier1"]
        tier2 = entry["relevant_pos"]["tier2"]

        for pi, profile in enumerate(PROFILES):
            qvec    = q_vecs[qi, pi, :].astype(np.float64)
            results = safe_search(aspace, gl, qvec, alpha=0.75)[:TOP_K]

            if not results:
                lambda_zero_count += 1

            m = score_single(results, tier1, tier2)

            rows.append(dict(
                variant         = label,
                entry_id        = entry["entry_id"],
                source_title    = entry["source_title"],
                source_category = entry["source_category"],
                category_size   = entry["category_size"],
                profile         = profile,
                **m,
            ))

    df = pd.DataFrame(rows)

    summary = df.groupby("profile").agg(
        hit_rate  = ("hit",          "mean"),
        mean_mrr  = ("mrr",          "mean"),
        mean_ndcg = ("ndcg",         "mean"),
        mean_rec2 = ("recall_tier2", "mean"),
        mean_gap  = ("score_gap",    "mean"),
        n         = ("entry_id",     "count"),
    ).round(3)

    print(f"\n  PER-PROFILE\n{summary.to_string()}")

    total = len(df)
    hits  = int(df["hit"].sum())
    print(f"\n  HEADLINE [{label}]")
    print(f"    Hit@{TOP_K}        : {df['hit'].mean():.3f}  ({hits}/{total})")
    print(f"    Mean MRR           : {df['mrr'].mean():.3f}")
    print(f"    Mean NDCG@{TOP_K}  : {df['ndcg'].mean():.3f}")
    print(f"    Recall@tier2       : {df['recall_tier2'].mean():.3f}")
    print(f"    Score gap          : {df['score_gap'].mean():.4f}")
    print(f"    Build time         : {build_t:.1f}s")
    print(f"    λ=0 queries        : {lambda_zero_count} / {total} "
          f"({lambda_zero_count/total:.3%})")

    return df

In [17]:
# ── resolve only highest-dim variants (d = 768) ──────────────────────────────
VARIANTS = [(extraction, d) for extraction, d in product(EXTRACTIONS, DIMS) if d == 768]
print(f"Running {len(VARIANTS)} variants: {VARIANTS}\n")

all_results: dict[str, pd.DataFrame] = {}

for extraction, d in VARIANTS:
    label       = f"nomic_{extraction}_{d}d"
    corpus_path = NOMIC_DIR / f"embeddings_nomic_{extraction}_{d}d.npy"
    query_path  = QUERY_DIR / f"queries_emb_nomic_{d}d.npy"

    if not corpus_path.exists():
        print(f"  ⚠ SKIP {label}: corpus not found ({corpus_path.name})")
        continue
    if not query_path.exists():
        print(f"  ⚠ SKIP {label}: queries not found ({query_path.name})")
        continue

    df = evaluate_variant(label, corpus_path, query_path, BQ)

    out_csv = OUT_DIR / f"eval_{label}_n{len(BQ)}.csv"
    df.to_csv(out_csv, index=False)
    print(f"\n  ✓ Saved → {out_csv.name}")

    all_results[label] = df

print(f"\n\nDone. Evaluated {len(all_results)}/{len(VARIANTS)} variants.")

Running 3 variants: [('compact', 768), ('full', 768), ('structured', 768)]


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  nomic_compact_768d
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Corpus shape : (20000, 768)
  Query shape  : (1379, 3, 768)
  Building ArrowSpace index...


  Graph params : {'eps': 0.75, 'k': 25, 'topk': 12, 'p': 2.0, 'sigma': None}
  Build time   : 83.35s

  PER-PROFILE
            hit_rate  mean_mrr  mean_ndcg  mean_rec2  mean_gap     n
profile                                                             
q_medium       0.780     0.492      0.365      0.065     0.063  1379
q_sentence     0.761     0.524      0.370      0.062     0.060  1379
q_verbose      0.717     0.496      0.354      0.059     0.056  1379

  HEADLINE [nomic_compact_768d]
    Hit@10        : 0.753  (3115/4137)
    Mean MRR           : 0.504
    Mean NDCG@10  : 0.363
    Recall@tier2       : 0.062
    Score gap          : 0.0599
    Build time         : 83.4s
    λ=0 queries        : 0 / 4137 (0.000%)

  ✓ Saved → eval_nomic_compact_768d_n1379.csv

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  nomic_full_768d
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Corpus shape : (20000, 768)
  Query shape  : (1379, 3, 768)
  Building ArrowSpace i

In [5]:
# ── resolve only highest-dim variants (d = 768) ──────────────────────────────
VARIANTS = [(extraction, d) for extraction, d in product(EXTRACTIONS, DIMS) if d == 512]
print(f"Running {len(VARIANTS)} variants: {VARIANTS}\n")

all_results: dict[str, pd.DataFrame] = {}

for extraction, d in VARIANTS:
    label       = f"nomic_{extraction}_{d}d"
    corpus_path = NOMIC_DIR / f"embeddings_nomic_{extraction}_{d}d.npy"
    query_path  = QUERY_DIR / f"queries_emb_nomic_{d}d.npy"

    if not corpus_path.exists():
        print(f"  ⚠ SKIP {label}: corpus not found ({corpus_path.name})")
        continue
    if not query_path.exists():
        print(f"  ⚠ SKIP {label}: queries not found ({query_path.name})")
        continue

    df = evaluate_variant(label, corpus_path, query_path, BQ)

    out_csv = OUT_DIR / f"eval_{label}_n{len(BQ)}.csv"
    df.to_csv(out_csv, index=False)
    print(f"\n  ✓ Saved → {out_csv.name}")

    all_results[label] = df

print(f"\n\nDone. Evaluated {len(all_results)}/{len(VARIANTS)} variants.")

Running 3 variants: [('compact', 512), ('full', 512), ('structured', 512)]


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  nomic_compact_512d
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Corpus shape : (20000, 512)
  Query shape  : (1379, 3, 512)
  Building ArrowSpace index...
  Graph params : {'eps': 0.55, 'k': 25, 'topk': 12, 'p': 2.0, 'sigma': None}
  Build time   : 39.05s

  PER-PROFILE
            hit_rate  mean_mrr  mean_ndcg  mean_rec2  mean_gap     n
profile                                                             
q_medium       0.666     0.412      0.321      0.058     0.061  1379
q_sentence     0.657     0.430      0.323      0.053     0.059  1379
q_verbose      0.643     0.429      0.316      0.054     0.054  1379

  HEADLINE [nomic_compact_512d]
    Hit@10        : 0.655  (2711/4137)
    Mean MRR           : 0.424
    Mean NDCG@10  : 0.320
    Recall@tier2       : 0.055
    Score gap          : 0.0578
    Build time         : 39.1s
   

In [6]:
# v2 baseline from previous full run (benchmark_queries.json, different benchmark)
# Use as directional reference only — benchmarks differ slightly
V2_BASELINE = dict(hit_rate=0.741, mean_mrr=0.518, mean_ndcg=0.376,
                   recall_tier2=0.172, score_gap=0.1302)

rows = []
for label, df in all_results.items():
    rows.append(dict(
        variant      = label,
        hit_rate     = round(df["hit"].mean(), 3),
        mean_mrr     = round(df["mrr"].mean(), 3),
        mean_ndcg    = round(df["ndcg"].mean(), 3),
        recall_tier2 = round(df["recall_tier2"].mean(), 3),
        score_gap    = round(df["score_gap"].mean(), 4),
    ))

# add baseline row for reference
rows.append({**{"variant": "── v2_baseline (ref) ──"}, **V2_BASELINE})

cmp = pd.DataFrame(rows).sort_values("hit_rate", ascending=False).reset_index(drop=True)
cmp["Δhit"] = (cmp["hit_rate"] - V2_BASELINE["hit_rate"]).round(3)

print("=" * 72)
print("  FULL COMPARISON TABLE (sorted by Hit@10)")
print("=" * 72)
display(cmp)

# best variant
best_label = cmp[~cmp["variant"].str.startswith("──")].iloc[0]["variant"]
print(f"\n  🏆  Best variant: {best_label}")

  FULL COMPARISON TABLE (sorted by Hit@10)


,variant,hit_rate,mean_mrr,mean_ndcg,recall_tier2,score_gap,Δhit
0,nomic_full_512d,0.805,0.551,0.379,0.060,0.0607,0.064
1,nomic_structured_512d,0.746,0.486,0.346,0.057,0.0561,0.005
2,── v2_baseline (ref) ──,0.741,0.518,0.376,0.172,0.1302,0.000
3,nomic_compact_512d,0.655,0.424,0.320,0.055,0.0578,-0.086



  🏆  Best variant: nomic_full_512d


In [19]:
best_df = all_results[best_label]

print(f"Per-profile breakdown — {best_label}")
print("=" * 50)

profile_summary = best_df.groupby("profile").agg(
    hit_rate  = ("hit",          "mean"),
    mean_mrr  = ("mrr",          "mean"),
    mean_ndcg = ("ndcg",         "mean"),
    mean_rec2 = ("recall_tier2", "mean"),
    mean_gap  = ("score_gap",    "mean"),
    n         = ("entry_id",     "count"),
).round(3)

display(profile_summary)

Per-profile breakdown — nomic_full_768d


,hit_rate,mean_mrr,mean_ndcg,mean_rec2,mean_gap,n
profile,,,,,,
q_medium,0.830,0.558,0.387,0.062,0.066,1379
q_sentence,0.811,0.575,0.389,0.061,0.062,1379
q_verbose,0.761,0.521,0.362,0.060,0.057,1379


In [20]:
cat = best_df.groupby("source_category").agg(
    hit_rate  = ("hit",          "mean"),
    mean_mrr  = ("mrr",          "mean"),
    mean_ndcg = ("ndcg",         "mean"),
    mean_rec2 = ("recall_tier2", "mean"),
    mean_gap  = ("score_gap",    "mean"),
    n         = ("entry_id",     "nunique"),
    cat_size  = ("category_size","first"),
).reset_index()

print(f"── Top 15 WORST categories (by hit_rate) — {best_label} ──")
display(cat.nsmallest(15, "hit_rate")[
    ["source_category","hit_rate","mean_mrr","mean_ndcg","mean_rec2","mean_gap","cat_size"]
].reset_index(drop=True))

print(f"\n── Top 15 BEST categories ──")
display(cat.nlargest(15, "hit_rate")[
    ["source_category","hit_rate","mean_mrr","mean_ndcg","mean_rec2","cat_size"]
].reset_index(drop=True))

── Top 15 WORST categories (by hit_rate) — nomic_full_768d ──


,source_category,hit_rate,mean_mrr,mean_ndcg,mean_rec2,mean_gap,cat_size
0,experimentation,0.000000,0.000000,0.105582,0.666667,0.049675,1
1,lab-work,0.000000,0.000000,0.000000,0.000000,0.063233,0
2,legal-research,0.000000,0.000000,0.000000,0.000000,0.033804,1
3,rubric,0.000000,0.000000,0.031197,0.111111,0.042786,3
4,task-prioritization,0.000000,0.000000,0.000000,0.000000,0.030065,2
5,customer-support,0.200000,0.081667,0.435564,0.005345,0.040097,898
6,competitive,0.333333,0.111111,0.166667,0.000000,0.068511,0
7,email-writing,0.333333,0.083333,0.143559,0.000000,0.027534,0
8,escalation-scripts,0.333333,0.288889,0.333890,0.057333,0.049801,50
9,hr-policies,0.333333,0.047619,0.078231,0.047619,0.026850,14



── Top 15 BEST categories ──


,source_category,hit_rate,mean_mrr,mean_ndcg,mean_rec2,cat_size
0,ai-design,1.0,0.169841,0.358798,0.000000,0
1,arch-viz,1.0,0.729167,0.505296,0.000000,2
2,audio-design,1.0,0.583333,0.687202,0.000000,0
3,automation,1.0,0.722222,0.306406,0.030303,11
4,balance,1.0,0.944444,0.640518,0.111111,2
5,benchmarking,1.0,0.400000,0.549571,0.000000,0
6,change-management,1.0,1.000000,1.000000,0.000000,0
7,chatbot,1.0,0.730556,0.379334,0.111111,7
8,chatbot-flow,1.0,0.791667,0.641295,0.000000,1
9,checklist,1.0,0.314815,0.362853,0.000000,1


In [21]:
print("Score gap distribution across variants")
print("Higher gap = ArrowSpace can discriminate more sharply\n")

gap_rows = []
for label, df in all_results.items():
    gap_rows.append(dict(
        variant  = label,
        mean_gap = round(df["score_gap"].mean(), 4),
        p25_gap  = round(df["score_gap"].quantile(0.25), 4),
        p75_gap  = round(df["score_gap"].quantile(0.75), 4),
        min_gap  = round(df["score_gap"].min(), 4),
        max_gap  = round(df["score_gap"].max(), 4),
    ))

gap_df = pd.DataFrame(gap_rows).sort_values("mean_gap", ascending=False)
display(gap_df)

print("\nNote: compare mean_gap vs hit_rate — if a variant has")
print("higher gap AND higher hit_rate, the Matryoshka structure")
print("is genuinely enriching the ArrowSpace λτ spectrum.")

Score gap distribution across variants
Higher gap = ArrowSpace can discriminate more sharply



,variant,mean_gap,p25_gap,p75_gap,min_gap,max_gap
1,nomic_full_768d,0.0619,0.0368,0.0811,0.0000,0.1974
0,nomic_compact_768d,0.0599,0.0353,0.0787,0.0065,0.2219
2,nomic_structured_768d,0.0593,0.0354,0.0772,0.0000,0.1906



Note: compare mean_gap vs hit_rate — if a variant has
higher gap AND higher hit_rate, the Matryoshka structure
is genuinely enriching the ArrowSpace λτ spectrum.
